In [ ]:
import numpy as np
import pandas as pd
import os
import glob

In [ ]:
os.getcwd()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df_PID = pd.read_csv('C:\\Users\\...\\HGPpid.csv')
df_PID.head()

In [ ]:
print(df_PID.shape)

In [ ]:
df_PID.dtypes
# dtype 'object' is used for strings
# int64 for 64bit integers

In [ ]:
# change dtype of Sample Number to object
df_PID['Sample Number'] = df_PID['Sample Number'].astype(str)
df_PID['NHS Number'] = df_PID['NHS Number'].astype(str)

In [ ]:
df_PID.dtypes

In [ ]:
df_PID['Sample Number'] = df_PID['Sample Number'].str.strip()
# str.strip should remove leading and trailing white spaces

In [ ]:
df_PID[df_PID.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')

In [ ]:
# NOTE that the sample numbers have had the preceding 0 stripped at read_csv
# So adding it back in here
df_PID['Sample Number'] = '0' + df_PID['Sample Number'].astype(str)

In [ ]:
# NOTE don't need columns Exam Code and Sample Type
# df_hgp = df_hgp.drop(labels=[5209, 5053, 5170, 6081, 6101], axis=0)
df_PID = df_PID.drop(['Exam Code', 'Sample Type'], axis=1)
print(df_PID.shape)

In [ ]:
df_PID.head()

In [ ]:
df_hgp = pd.read_csv('C:\\Users\\grim0006\\DataNow\\Home\\HSST\\C1 C2\\HGP Genotype data\\df_outputs\\HGPgeno_data_ready_for_merge.csv')
df_hgp.head()

In [ ]:
print(df_hgp.shape)

In [ ]:
df_hgp.dtypes

In [ ]:
df_hgp['Sample Number'] = df_hgp['Sample Number'].str.strip()
# str.strip should remove leading and trailing white spaces

In [ ]:
df_hgp[df_hgp.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')

In [ ]:
# these were not previously detected as duplicates when cleaning the data
# possibly the .str.strip() command or the removal of the check digit - upper v lower case?
# anyhow, duplicates at Sample Number are duplicates throughout the genotype
# so can remove one of each duplicate pair
df_hgp = df_hgp.drop(labels=[5209, 5053, 5170, 6081, 6101], axis=0)
print(df_hgp.shape)
# should probably use .drop_duplicated, keep='First'

In [ ]:
df_hgp.head()

In [ ]:
df_project_left = pd.merge(df_hgp, df_PID, how='left', on="Sample Number")
print(df_project_left.shape)

In [ ]:
print(df_project_left.isnull().sum())
# 6621 - 392 = 6229 complete rows

In [ ]:
df_project_left.head()

In [ ]:
# df_project_left.to_csv('df_project_left_missing_data.csv',index=False)

In [ ]:
df_project_right = pd.merge(df_hgp, df_PID, how='right', on="Sample Number")
print(df_project_right.shape)

In [ ]:
print(df_project_right.isnull().sum())
# 6965 - 736 = 6229 complete rows

In [ ]:
df_project_right.head()

In [ ]:
# df_project_right.to_csv('df_project_right_missing_data.csv',index=False)

In [ ]:
# merge left and right result in 6229 complete rows
# can use either to drop NaN rows
df_project = df_project_right.dropna(axis=0)
print(df_project.shape)

In [ ]:
df_project[df_project.duplicated('NHS Number', keep=False) == True].sort_values('NHS Number')

In [ ]:
# The hgp data was cleaned to retain only full genotypes
# the PID data was cleaned but always had duplicates of patients
# this was necessary, as it was possible that only one of the sample numbers would have had a full genotype
# now need to pick out samples that are duplicates over the NHS number and genotype
df_project.drop_duplicates(subset=['NHS Number', 'RH01G', 'RH02G', 'RH03G', 'RH04G', 'RH05G', 'RH10G', 'RH20G', 'RH19G', 
                                   'RH31G', 'RHD455', 'RHD_INT4', 'RHD667',	'EX5', 'RHC', 'RH_EX7', 'RHE', 'RHCE667', 
                                   'RHCE1006',	'RHCE712', 'RHCE733'],
                         keep='first', inplace=True)
print(df_project.shape)

In [ ]:
6229-5924
# number of duplicates across all those coulmns, now dropped:

In [ ]:
df_dups = df_project[df_project.duplicated('NHS Number', keep=False) == True].sort_values('NHS Number')

In [ ]:
print(df_dups.shape)

In [ ]:
df_dups

In [ ]:
# df_dups represents pateints with different results on different samples
# the most recently tested sample will be retained
# first sort by Sample Number as these are numerically chronological
df_dups = df_dups.sort_values(by=['Sample Number'])

In [ ]:
# then, drop_duplicates keep=last
df_dups.drop_duplicates(subset=['NHS Number'], keep='last', inplace=True) 

In [ ]:
df_dups

In [ ]:
# now drop duplicates at NHS number from df_project
# note keep=false ensures all duplicates at NHS number are dropped
df_project.drop_duplicates(subset=['NHS Number'], keep=False, inplace=True)
print(df_project.shape)

In [ ]:
5924-56
# there were 56 duplicates at NHS number

In [ ]:
# now df_project contains none of duplicates at NHS number
# concat df_project with df_dups (df_dups contains only the most recent duplicate)
# df = pd.concat([df_project_minusone, df_project_one], ignore_index=True, sort=False)
# print(df.shape)
df_project2 = pd.concat([df_dups, df_project], ignore_index=True, sort=False)

In [ ]:
print(df_project2.shape)

In [ ]:
5896-5868
# = 28 = 56/2 = correct

In [ ]:
# renaming df_project2 here, back to df_project to ensure I don;t have to amend below code
df_project2 = df_project

In [ ]:
# The HGP nomenclature of using the nucleotide change e.g. G/G, G/C or C/C is
# complicated because Wild Type is not always first.
# A better system is 0/0  0/1 and 1/1
# It is now necesary to edit the genotypes to the numerical nomenclature
df_project['RHD455'] = df_project['RHD455'].replace('A/A', '0/0')
df_project.head()
# Error but it still works - strange
# carry on for now and consider 'fixing' later/
# I don't think .loc will work as I don't know all the locations?

In [ ]:
# This now needs repeating for each possible genotype for RHD455
df_project['RHD455'] = df_project['RHD455'].replace('C/A', '0/1')
df_project['RHD455'] = df_project['RHD455'].replace('C/C', '1/1')
df_project.head(10)

In [ ]:
# The genotype nomenclature now needs replacing for other columns
# remember 'NOAMP' is also a legitimate call
# RHD_INT4 is unusual in that the HGP uses this as a marker for presence/absence
# of RHD and therefore 'C/C' and 'NOAMP' are the only expected calls.
df_project['RHD_INT4'] = df_project['RHD_INT4'].replace('C/C', 'AMP')

# RH_EX7 is also unusual
# A/A indicates no RHD gene is present; copy number 0
# A/T indiactes one copy of RHD is probably present; copy number 1
# T/T indicates two copies are probably present; copy number 2
# there are limitations see DAT2812
df_project['RH_EX7'] = df_project['RH_EX7'].replace('T/T', 'two')
df_project['RH_EX7'] = df_project['RH_EX7'].replace('A/T', 'one')
df_project['RH_EX7'] = df_project['RH_EX7'].replace('A/A', 'zero')

# RHD667 causes a variant call
df_project['RHD667'] = df_project['RHD667'].replace('T/T', '0/0')
df_project['RHD667'] = df_project['RHD667'].replace('G/T', '0/1')
df_project['RHD667'] = df_project['RHD667'].replace('G/G', '1/1')

# 'EX5' is indicative of the RHD pseudogene
df_project['EX5'] = df_project['EX5'].replace('C/C', '0/0')
df_project['EX5'] = df_project['EX5'].replace('T/C', '0/1')
df_project['EX5'] = df_project['EX5'].replace('T/T', '1/1')

# 'RHC' 0,0 represents little c
df_project['RHC'] = df_project['RHC'].replace('C/C', '0/0')
df_project['RHC'] = df_project['RHC'].replace('T/C', '0/1')
df_project['RHC'] = df_project['RHC'].replace('T/T', '1/1')

# 'RHE' 0,0 represents little e
df_project['RHE'] = df_project['RHE'].replace('G/G', '0/0')
df_project['RHE'] = df_project['RHE'].replace('G/C', '0/1')
df_project['RHE'] = df_project['RHE'].replace('C/C', '1/1')

# RHCE667 causes a cariant phenotype call
df_project['RHCE667'] = df_project['RHCE667'].replace('G/G', '0/0')
df_project['RHCE667'] = df_project['RHCE667'].replace('T/G', '0/1')
df_project['RHCE667'] = df_project['RHCE667'].replace('T/T', '1/1')

df_project['RHCE1006'] = df_project['RHCE1006'].replace('G/G', '0/0')
df_project['RHCE1006'] = df_project['RHCE1006'].replace('G/T', '0/1')
df_project['RHCE1006'] = df_project['RHCE1006'].replace('T/T', '1/1')

df_project['RHCE712'] = df_project['RHCE712'].replace('A/A', '0/0')
df_project['RHCE712'] = df_project['RHCE712'].replace('G/A', '0/1')
df_project['RHCE712'] = df_project['RHCE712'].replace('G/G', '1/1')

df_project['RHCE733'] = df_project['RHCE733'].replace('C/C', '0/0')
df_project['RHCE733'] = df_project['RHCE733'].replace('G/C', '0/1')
df_project['RHCE733'] = df_project['RHCE733'].replace('G/G', '1/1')

df_project.head(10)

In [ ]:
# amend apparent homozygous RHD calls in copy number 1 (RH_EX7) to hemizygous.
# IF RH_EX7 = 'one', change values in RHD455, EX5 and RHD667 from '0/0' to '0' and from '1/1' to '1'
# 0/1 calls should not exist (or be very rare) in RHD667 and EX5
# RHD455 will return 0/1 calls more often when RH_EX7 = one due to hybrid alleles
# first creat a slice containing only RH_EX7 = one results
df_project_one = df_project[df_project['RH_EX7'].str.contains('one')]
df_project_one.head()

In [ ]:
print(df_project_one.shape)

In [ ]:
df_project_one['RHD667'] = df_project_one['RHD667'].replace('0/0', '0')
df_project_one['RHD667'] = df_project_one['RHD667'].replace('1/1', '1')

df_project_one['EX5'] = df_project_one['EX5'].replace('0/0', '0')
df_project_one['EX5'] = df_project_one['EX5'].replace('1/1', '1')

df_project_one['RHD455'] = df_project_one['RHD455'].replace('0/0', '0')
df_project_one['RHD455'] = df_project_one['RHD455'].replace('1/1', '1')

df_project_one.head()

In [ ]:
# now re-combine the slice df_project_one back into df_project

# df_project_right = pd.merge(df_hgp, df_PID, how='right', on="Sample Number")
# print(df_project_right.shape)
# merge didn't work!

# try deleting all rows with RH_EX7 = one from df_project
# df_project_minusone = df_project.drop['RH_EX7'].str.contains('one')
# df = df.drop(df[df['city'] == 'Chicago'].index)
df_project_minusone = df_project.drop(df_project[df_project['RH_EX7'] == 'one'].index)
df_project_minusone.head()

In [ ]:
print(df_project_minusone.shape)

In [ ]:
# value counts on RHEX7
RH_EX7counts_minusone = df_project_minusone['RH_EX7'].value_counts()
print(RH_EX7counts_minusone)

In [ ]:
3834+2034

In [ ]:
df = pd.concat([df_project_minusone, df_project_one], ignore_index=True, sort=False)
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# counts = df['A'].value_counts()
RH_EX7counts = df['RH_EX7'].value_counts()
print(RH_EX7counts)

In [ ]:
# Not possible for RH_EX7 to be G/G
# find and delete this row
df = df.drop(df[df['RH_EX7'] == 'G/G'].index)
RH_EX7counts = df['RH_EX7'].value_counts()
print(RH_EX7counts)

In [ ]:
# unusually, it is also not possible for RH_EX7 to be NOAMP
# this is because RH-EX7 will always amplify RHCE in the absence of RHD
# find and delete these rows
df = df.drop(df[df['RH_EX7'] == 'NOAMP'].index)
RH_EX7counts = df['RH_EX7'].value_counts()
print(RH_EX7counts)

In [ ]:
RHD455counts = df['RHD455'].value_counts()
print(RHD455counts)

In [ ]:
RHD_INT4counts = df['RHD_INT4'].value_counts()
print(RHD_INT4counts)

In [ ]:
EX5counts = df['EX5'].value_counts()
print(EX5counts)

In [ ]:
RHD667counts = df['RHD667'].value_counts()
print(RHD667counts)

In [ ]:
RH01Gcounts = df['RH01G'].value_counts()
print(RH01Gcounts)

In [ ]:
# all the counts in the RHD columns are genuine = clean
print(df.shape)

In [ ]:
# now need to re-organise the column order so that strange genotpes can be viewed.
# first, re-generate the list of column titles
column_names = list(df.columns.values)
print(column_names)

In [ ]:
# then copy these, into this function and re-order
df = df[['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'RH01G', 
                         'RHC', 'RHE', 'RHCE667', 'RHCE1006', 'RHCE712', 'RHCE733', 'RH02G', 'RH03G', 'RH04G', 'RH05G', 
                         'RH10G', 'RH20G', 'RH19G', 'RH31G', 
                         'Sample Number', 'Patient ID', 'NHS Number', 'First Name', 'Surname', 'DOB', 'Referring Hospital', 'source']]
df.head()

In [ ]:
df = df.sort_values(by=['RHD_INT4', 'EX5', 'RHD455', 'RHD667', 'RH_EX7', 'RH01G'])
# pd.set_option('display.max_rows', None)
df.head()

In [ ]:
# end here for this script
# the df above has no duplicate samples and only complete and valid genotypes across RH
# However, it does have rows that give different D predicted phenotypes for the same genotype
# this is because of changes in the TT in HGP in ~2019
# subsequent scripts will resolve this for D and CE

In [ ]:
# df.to_csv('merged_ready_to_delete_dup_geno_calls.csv',index=False)